<a href="https://colab.research.google.com/github/alexandrufalk/tensorflow/blob/Master/Session2_AI_convention_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LEARNING AI- Workshop: Demystifying Generative AI models - A hands-on approach notebook -2

> **⚠️ Disclaimer:** *Using LLMs is subject to approvals.*

In [ ]:
import os
path = os.path.join(os.getcwd(),'work')
# Relative path to the .env
dotenv_path = os.path.join(path, ".env")

In [ ]:
%pip install -r "{path}"/requirements.txt

In [ ]:
# Import necessary modules
from dotenv import load_dotenv
from typing import List, Sequence

# Import classes related to messages and prompts
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Import OpenAI wrapper for Azure
from langchain_openai import AzureChatOpenAI

# Import message graph utilities
from langgraph.graph import END, MessageGraph

# Load environment variables from a .env file
load_dotenv(dotenv_path)
os.environ["OPENAI_API_VERSION"] = "2023-12-01-preview"
os.environ["AZURE_OPENAI_ENDPOINT"] = os.getenv('AZURE_OPENAI_ENDPOINT')
os.environ["AZURE_OPENAI_API_KEY"] = os.getenv('AZURE_OPENAI_API_KEY')
os.environ['REQUESTS_CA_BUNDLE'] = os.path.join(path, 'certificates.pem')
deployment=os.getenv('MODEL_ID') # deployment='gpt-4o-mini'

# Introduction to prompting and examples

<img src="./work/session_2/prompt.png" alt="Alt text" width="1000" height="900">


In [ ]:
# First code o start
from openai import AzureOpenAI

# Initialize the AzureOpenAI client.
# The AzureOpenAI client allows us to interact with the Azure-hosted OpenAI services.
client = AzureOpenAI(
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"),
  api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-08-01-preview",
)


In [ ]:
message_automotive = [
    {
        "role": "system",
        "content": (
            "You are an automotive assistant, you must technically provide your answer only related to automotive. "
            "Instructions: "
            "- If the query is not related to automotive, you can say 'I do not know' or 'I am not sure'."
        )
    },
    {
        "role": "user",
        # Related to automotive
        # "content": "what is ADAS?"
        "content": "Can you give me an overview of current safety features of cars?"

        # Not related to automotive
        # "content": "what is software development?"
    }
]


In [ ]:
response = client.chat.completions.create(
    # model = "deployment_name".
    model=deployment,
    messages=message_automotive,
    temperature=0,
    max_tokens=200,
)
print(response.choices[0].message.content)

Certainly! Current safety features in cars have advanced significantly and can be categorized into several key areas:

1. **Active Safety Features**:
   - **Anti-lock Braking System (ABS)**: Prevents wheel lock-up during braking, allowing for better steering control.
   - **Electronic Stability Control (ESC)**: Helps prevent skidding and loss of control by automatically applying brakes to individual wheels.
   - **Traction Control System (TCS)**: Reduces wheel spin during acceleration by adjusting engine power or applying brakes.
   - **Adaptive Cruise Control (ACC)**: Maintains a set speed while automatically adjusting to the speed of the vehicle ahead.
   - **Lane Departure Warning (LDW) and Lane Keeping Assist (LKA)**: Alerts drivers when they unintentionally drift out of their lane and can provide steering assistance to keep the vehicle in its lane.

2. **Passive Safety Features**:
   - **Airbags**: Front, side, curtain, and knee airbags deploy in


# The Basic Ingredients of a Prompt

<img src="./work/session_2/basic_prompting.png" alt="Alt text" width="1000" height="900">


# The Potential Complexity of a Prompt

<img src="./work/session_2/complex_prompt.png" alt="Alt text" width="800" height="700">


In [ ]:
#example
# Prompt components
persona = "You are an expert in Large Language models. You excel at breaking down complex papers into digestible summaries.\n"
instruction = "Summarize the key findings of the paper provided.\n"
context = "Your summary should extract the most crucial points that can help researchers quickly understand the most vital information of the paper.\n"
data_format = "Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.\n"
audience = "The summary is designed for busy researchers that quickly need to grasp the newest trends in Large Language Models.\n"
tone = "The tone should be professional and clear.\n"
data = "Text to summarize: PUT_THE_DATA_TO_SUMMARIZE_HERE"
# The full prompt - remove and add pieces to view its impact on the generated output
query = persona + instruction + context + data_format + audience + tone + data
# prompt = format_prompt(query)
print(query)

You are an expert in Large Language models. You excel at breaking down complex papers into digestible summaries.
Summarize the key findings of the paper provided.
Your summary should extract the most crucial points that can help researchers quickly understand the most vital information of the paper.
Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.
The summary is designed for busy researchers that quickly need to grasp the newest trends in Large Language Models.
The tone should be professional and clear.
Text to summarize: PUT_THE_DATA_TO_SUMMARIZE_HERE


### Incase if you want the response as dictionary

In [ ]:
import pprint
output_dict = response.model_dump()
pprint.pprint(output_dict)

{'choices': [{'content_filter_results': {'hate': {'filtered': False,
                                                  'severity': 'safe'},
                                         'protected_material_code': {'detected': False,
                                                                     'filtered': False},
                                         'protected_material_text': {'detected': False,
                                                                     'filtered': False},
                                         'self_harm': {'filtered': False,
                                                       'severity': 'safe'},
                                         'sexual': {'filtered': False,
                                                    'severity': 'safe'},
                                         'violence': {'filtered': False,
                                                      'severity': 'safe'}},
              'finish_reason': 'length',
              'index':

### Get the token counts, which translate into costs

In [ ]:
# Find the the token and costing
print(output_dict['usage'])

{'completion_tokens': 200, 'prompt_tokens': 69, 'total_tokens': 269, 'completion_tokens_details': None, 'prompt_tokens_details': None}


### Controlling Response Randomness: Temperature

**Temperature:**
 - Control on determinism.
 -  Ranges from 0 (highly deterministic) to 2 (very random).




In [ ]:
# Define the prompt
prompt = "Describe a sports car driving down a highway."

# Set low and high temperature values
low_temp = 0.0
high_temp = 0.9

# Function to generate a response with a specified temperature
def generate_response(temperature):
    response = client.chat.completions.create(
        model='gpt-35-turbo',
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=200
    )
    return response.choices[0].message.content

# Generate responses with different temperature values
low_temp_response = generate_response(low_temp)
high_temp_response = generate_response(high_temp)

# Print the responses
print(f'Temperature = {low_temp}, Response: {low_temp_response}')


Temperature = 0.0, Response: As the sleek sports car zoomed down the highway, the roar of its powerful engine echoed through the air. The car hugged the road tightly, its low profile and aerodynamic design allowing it to effortlessly slice through the wind. The driver's hands gripped the steering wheel tightly, expertly maneuvering the car through the twists and turns of the road. The car's headlights illuminated the way ahead, casting a bright beam of light on the asphalt. The rush of adrenaline was palpable as the car accelerated, leaving other vehicles in its wake. The wind whipped through the driver's hair, adding to the thrill of the ride. The car's exhaust emitted a deep, throaty growl as it continued to speed down the highway, leaving a trail of dust and excitement in its wake.


In [ ]:
print(f'\nTemperature = {high_temp}, Response: {high_temp_response}')


Temperature = 0.9, Response: The sleek, low-slung body of the sports car seemed to hug the road as it sped down the highway, effortlessly weaving in and out of traffic. The powerful engine roared as it accelerated, sending a rush of adrenaline through the driver and passengers alike. The wind whipped through their hair, the windows down as they reveled in the thrill of the speed. The sun glinted off the polished exterior, making the car seem almost otherworldly in its beauty. As it zoomed past other vehicles, the car seemed to leave a trail of awe in its wake, a testament to the power and allure of the sports car.


# Prompting with LangChain

## What is LangChain?
- LangChain is a framework designed to build applications using language models.
- It helps in managing prompts, chains of calls, and agent logic.
- Provides a seamless way to integrate various LLMs (Large Language Models) and API calls.

## Why Use Prompting with LangChain?
- **Simplifies Prompt Management**: Centralizes prompt templates, making it easier to update or switch out prompts.
- **Structured Approach**: Helps in creating complex chains of prompts that interact with different APIs or models.
- **Enhanced Flexibility**: Customizes responses from LLMs to better align with specific use cases, like generating reports, answering questions, or analyzing data.
  
## Key Concepts
- **Prompt Templates**: Define the structure of prompts, making them reusable and adaptable to different inputs.
- **Chains**: A series of calls or prompts that can be executed in sequence or in a decision-based manner.
- **Agents**: Components that can make decisions and dynamically interact with external tools or APIs based on prompts.




 <img src="./work/session_2/Langchain.png" alt="Alt text" width="1000" height="900">


### Example using PromptTemplate

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser


In [ ]:
# Set up a parser + inject instructions into the prompt template.
parser = JsonOutputParser()

In [ ]:
template = "You are an automotive expert, answer the question.\n{format_instructions}\n {question}"
prompt_template = PromptTemplate(
    template=template,
    input_variables=["question"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)


In [ ]:
print(prompt_template.invoke({"question": "What is ABS"}))

text='You are an automotive expert, answer the question.\nReturn a JSON object.\n What is ABS'


In [ ]:
# Let's integrate with LLM
llm = AzureChatOpenAI(
    # openai_api_version="2024-07-18",
    azure_deployment=deployment,)

In [ ]:
llm

AzureChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x7f94d18cb710>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x7f94d16ed9d0>, root_client=<openai.lib.azure.AzureOpenAI object at 0x7f94d1968c90>, root_async_client=<openai.lib.azure.AsyncAzureOpenAI object at 0x7f94d1968c50>, model_kwargs={}, openai_api_key=SecretStr('**********'), disabled_params={'parallel_tool_calls': None}, azure_endpoint='https://ai-convention-workshop-2024.openai.azure.com/', deployment_name='gpt-4o-mini', openai_api_version='2023-12-01-preview', openai_api_type='azure')

### Chaining in Langchain

In [ ]:
# Langchain expression language (LCEL)
# Connect calls to different component -prompt_template and LLM
llm_chain = prompt_template | llm
question = "What is ABS"
llm_chain.invoke({"question": question})

AIMessage(content='```json\n{\n  "ABS": {\n    "full_form": "Anti-lock Braking System",\n    "definition": "A safety system in vehicles that prevents the wheels from locking up during braking, allowing the driver to maintain steering control.",\n    "components": [\n      {\n        "name": "Wheel Speed Sensors",\n        "function": "Monitor the speed of each wheel and provide data to the ABS control module."\n      },\n      {\n        "name": "ABS Control Module",\n        "function": "Processes data from wheel speed sensors and determines whether to modulate brake pressure."\n      },\n      {\n        "name": "Hydraulic Control Unit (HCU)",\n        "function": "Adjusts brake pressure to each wheel to prevent locking and maintain traction."\n      }\n    ],\n    "benefits": [\n      "Improves vehicle control during hard braking.",\n      "Reduces stopping distances on slippery surfaces.",\n      "Prevents skidding and loss of steering control."\n    ],\n    "common_issues": [\n   

In [ ]:
# Langchain expression language (LCEL)
# connect calls to different component --prompt_template, LLM, output parser
llm_chain = prompt_template | llm | parser
question = "What is ABS"
print(llm_chain.invoke({"question": question}))

{'ABS': {'full_name': 'Anti-lock Braking System', 'definition': 'A safety system in vehicles that prevents the wheels from locking up during braking.', 'function': {'primary_purpose': 'To maintain steering control during hard braking.', 'mechanism': {'sensors': 'Monitors wheel speed.', 'valves': 'Regulates brake pressure.', 'controller': 'Modulates braking force to prevent wheel lock-up.'}}, 'benefits': ['Improved vehicle control during emergency stops.', 'Reduced stopping distances on slippery surfaces.', 'Prevention of skidding and loss of traction.'], 'history': {'introduction': 'First developed in the 1920s and became common in passenger vehicles in the 1980s.', 'regulatory': 'Mandated in many countries for new vehicles due to safety standards.'}, 'types': ['Four-channel ABS', 'Three-channel ABS', 'Two-channel ABS', 'One-channel ABS']}}


# Advance prompting technique

## In-Context Learning

In more complex scenarios, models may lack the necessary understanding or context to generate an appropriate response based solely on a given prompt. In such cases, it is beneficial to provide examples to the model, enabling it to learn and adapt from these examples—a process known as *in-context learning*.


### Few short prompting

<img src="./work/session_2/few_shot.png" alt="Alt text" width="800" height="600">


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate


In [ ]:
# source for reviews : https://www.amazon.in/Urban-Terrain-Performance-Mountain-Suspension/dp/B0CVH8LGTZ/ref=zg_bs_g_3403870031_d_sccl_1/262-5569152-5785625?th=1

examples = [
    {"input" : "The cycle is smooth and easy to ride and especially the wheel is strong and can ride in tough terrain.. Installation person was knowledgeable and on time/discipline.", "output": "positive"},
    {"input" : "Don't buy above 150 height of men because it's very small cycle for child 10 year and I faced many issue. bearing jammed and small wheel but overall good for children. bult quality is so good.", "output":"negative" },
    {"input" : "The Cycle is good and its up and running, but the package / box was torn, open from bottom. looked like it was lying in the warehouse for a long time. had dust and dirt all over it. screws of fenders and other parts were missing. We got them fixed from the cycle shop. but overall the product is good and did not have damage to itself", "output":"neutral"},
    {"input" : "it is a very nice cycle with quality parts and reliable service. highly recommended if u want a good cycle in medium range price", "output": "positive"}
]
example_prompt = ChatPromptTemplate.from_messages([("human", "{input}"), ("ai", "{output}")])
few_shot_prompt = FewShotChatMessagePromptTemplate(example_prompt=example_prompt, examples= examples)
final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "your a bycycle reviewer who will give sentiment of the review for user input "),
        # uncomment for illustration
        # few_shot_prompt,
        ("human", "{input}"),
    ]
)

In [ ]:
chain = final_prompt | llm
res = chain.invoke("""it is a very nice cycle with quality parts and reliable service.highly recommended if u want a good cycle in medium range price.""")
print(res.content)

The sentiment of this review is positive. The reviewer highlights the quality of the bicycle, the reliability of the service, and recommends it for those looking for a good bike in a medium price range. Overall, it's an enthusiastic endorsement!


### Chain of thought prompting


<img src="./work/session_2/chain_of_thought.png" alt="Alt text" width="800" height="600">


In [ ]:
# Define the question
question = """
I went to the market and bought 10 apples. I gave 2 apples to the neighbor and 2 to the repairman.
I then went and bought 5 more apples and ate 1.
How many apples did I remain with?

Let's think step by step.
"""

# Run the model to get the response
response = llm.invoke(question)

# Print the model's response
print(response.content)

Let's break down the steps:

1. You initially bought 10 apples.
2. You gave 2 apples to the neighbor: 
   - 10 - 2 = 8 apples remaining.
3. You gave 2 apples to the repairman: 
   - 8 - 2 = 6 apples remaining.
4. You bought 5 more apples: 
   - 6 + 5 = 11 apples remaining.
5. You ate 1 apple: 
   - 11 - 1 = 10 apples remaining.

So, you have **10 apples** remaining.


### Self-Consistency


<img src="./work/session_2/self.consistancy.png" alt="Alt text" width="1000" height="900">


In [ ]:

# Define the problem statement
problem = "If there are 10 cars in the parking lot and 3 more cars arrive. Half the original number of cars leave. Then, half of the current number of cars arrive. How many cars are there in the parking?"

# Define a prompt template that simulates three independent experts reasoning
template = """
Imagine three completely independent experts who reason differently are answering this question.
Each expert will solve the problem step by step, and the final answer is obtained by majority vote.

Problem: {problem}

Expert 1: Let's think through this step by step.

Expert 2: Let's consider each part of the problem carefully.

Expert 3: I will analyze the situation and calculate the answer step by step.

Final answer:
"""

# Create a PromptTemplate with the input variable and template text
prompt = PromptTemplate(
    input_variables=["problem"],
    template=template
)

# Create the LLMChain with the language model and the prompt template
Self_Consistenc_chain = prompt | llm

# Run the LLMChain with the problem as input
response = Self_Consistenc_chain.invoke( problem)

# Print the result
print(response.content)


Let's go through the problem step by step with each expert's reasoning.

**Expert 1:**
1. Start with 10 cars in the parking lot.
2. 3 more cars arrive: 10 + 3 = 13 cars.
3. Half the original number of cars leave: Half of 10 is 5, so 13 - 5 = 8 cars remain.
4. Now, half of the current number of cars arrive: Half of 8 is 4, so 8 + 4 = 12 cars.
   
Final answer from Expert 1: **12 cars**

---

**Expert 2:**
1. We begin with 10 cars.
2. When 3 cars arrive, we have: 10 + 3 = 13 cars.
3. From the original 10 cars, half leave: 10 / 2 = 5 cars leave. Remaining cars: 13 - 5 = 8.
4. Now, half of the current 8 cars arrive: 8 / 2 = 4 cars arrive. Total now: 8 + 4 = 12 cars. 

Final answer from Expert 2: **12 cars**

---

**Expert 3:**
1. Initial number of cars is 10.
2. Three more cars arrive: 10 + 3 = 13 cars.
3. Half of the original 10 cars leave (which is 5): 13 - 5 = 8 cars left.
4. Then, we have half of the current number of cars (8) arriving: 8 / 2 = 4 cars arrive. Total: 8 + 4 = 12 cars.

F

## Tool Calling with LangChain
- Large Language Models (LLMs) can interact with external data sources via **tool calling functionality**.
- Tool calling is a powerful technique that enables:
    - **Developers to build sophisticated applications** leveraging LLMs.
    - **Access, interaction, and manipulation of external resources** like:
        - Databases
        - Files
        - APIs


In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply a and b."""
    return a * b

tools = [multiply]

llm_with_tools = llm.bind_tools(tools)

In [ ]:
result_1 = llm_with_tools.invoke("Hello world!")
print(result_1.content)

Hello! How can I assist you today?


In [ ]:
result_1.tool_calls


[]

In [ ]:
result_2 = llm_with_tools.invoke("What is 2 times 3?")
result_2

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_aEn1hAN7HghSbMCFaholQQry', 'function': {'arguments': '{"a":2,"b":3}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 53, 'total_tokens': 70, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_d54531d9eb', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], 'finish_reason': 'tool_calls', 'logprobs': None, 'content_filter_results': {}}, id='run-d026b353-9f16-4ca6-a468-3f97e896ca67-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'call_aEn1hAN7HghSbMCFaholQQry', '

In [ ]:
# Run the tool.

tool_call = result_2.tool_calls[0]

eval(tool_call['name']).func(**tool_call['args'])

6

In [ ]:
result_2.tool_calls


[{'name': 'multiply',
  'args': {'a': 2, 'b': 3},
  'id': 'call_aEn1hAN7HghSbMCFaholQQry',
  'type': 'tool_call'}]

# LangGraph


 LangGraph is built on top of LangChain and completely interoperable with the LangChain ecosystem. It adds new value primarily through the introduction of an easy way to create cyclical graphs. This is often useful when creating agent runtime

<img src="./work/session_2/Langgraph.png" alt="Alt text" width="800" height="600">


- LangChain enables easy creation of custom chains using LangChain Expression Language. Currently, chains function as directed acyclic graphs (DAGs) without cycles.
- Cycles in complex LLM applications allow LLMs to reason and determine next steps. Running an LLM in a loop enhances flexibility and handles more ambiguous use-cases.
- Agents operate in a loop, with the LLM determining actions and responses iteratively.
  LangGraph enables creation of state machines, allowing for controll


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, END

from langgraph.graph.message import add_messages, AnyMessage

class AgentState(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list[AnyMessage], add_messages]

In [ ]:
graph_builder = StateGraph(AgentState)

### Initilize the LLM and chatnode

In [ ]:
llm = AzureChatOpenAI(
    openai_api_version="2024-02-01",
    azure_deployment=deployment,)
def chat(state: AgentState):
    return {"messages": [llm.invoke(state["messages"])]}

graph_builder.add_node("chat", chat)

In [ ]:
graph_builder.set_entry_point("chat")
graph_builder.set_finish_point("chat")
graph_ = graph_builder.compile(checkpointer=MemorySaver())


### Visualize the graph

In [ ]:
graph = graph_builder.compile()
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(e)

Could not find a suitable TLS CA certificate bundle, invalid path: /home/jovyan/work/work/certificates.pem


In [ ]:
# Run the chatbot
# To end the chat, type "quit", "exit", or "q" or click the "Interrupt the kernel" button (Stop symbol) in the command panel at the top of this notebook.

while True:
    user_input = input("User: ")
    if user_input.lower() in ["quit", "exit", "q"]:
        print("Goodbye!")
        break
    config = {"configurable": {"thread_id": "1"}}
    for event in graph_.stream({"messages": [("user", user_input)]}, config=config):
        for value in event.values():
            print("Assistant:", value["messages"][-1].content)

User:  hi


Assistant: Hello again! How can I help you today?


User:  q


Goodbye!


## AI Agents with Agent_supervisor
## Advanced Topics


### Helper Utilities

Define a helper function below, which make it easier to add new agent worker nodes.

In [ ]:
from typing import Annotated,List
import random

from langchain.agents import AgentExecutor, create_openai_tools_agent


In [ ]:
def create_agent(llm: AzureOpenAI, tools: list, system_prompt: str):
    # Each worker node will be given a name and some tools.
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                system_prompt,
            ),
            MessagesPlaceholder(variable_name="messages"),
            MessagesPlaceholder(variable_name="agent_scratchpad"),
        ]
    )
    agent = create_openai_tools_agent(llm, tools, prompt)
    executor = AgentExecutor(agent=agent, tools=tools)
    return executor

In [ ]:
#function that will use to be the nodes in the graph - it takes care of converting the agent response to a human message. This is important because that is how we will add it the global state of the graph
def agent_node(state, agent, name):
    result = agent.invoke(state)
    return {"messages": [HumanMessage(content=result["output"], name=name)]}

### Create Agent Supervisor

It will use function calling to choose the next worker node OR finish processing.

In [ ]:
from langchain_core.output_parsers.openai_functions import JsonOutputFunctionsParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

members = ["weather_agent","similarity_agent"]

system_prompt = (
    "You are a supervisor tasked with managing a conversation between the"
    " following workers:  {members}. Given the following user request,"
    " respond with the worker to act next. Each worker will perform a"
    " task and respond with their results and status. When finished,"
    " respond with FINISH."
)
# Our team supervisor is an LLM node. It just picks the next agent to process
# and decides when the work is completed
options = ["FINISH"] + members
# Using openai function calling can make output parsing easier for us
function_def = {
    "name": "route",
    "description": "Select the next role.",
    "parameters": {
        "title": "routeSchema",
        "type": "object",
        "properties": {
            "next": {
                "title": "Next",
                "anyOf": [
                    {"enum": options},
                ],
            }
        },
        "required": ["next"],
    },
}
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="messages"),
        (
            "system",
            "Given the conversation above, who should act next?"
            " Or should we FINISH? Select one of: {options}",
        ),
    ]
).partial(options=str(options), members=", ".join(members))

supervisor_chain = (
    prompt
    | llm.bind_functions(functions=[function_def], function_call="route")
    | JsonOutputFunctionsParser()
)

/tmp/ipykernel_2863/1867772497.py:48: LangChainDeprecationWarning: The method `BaseChatOpenAI.bind_functions` was deprecated in langchain-openai 0.2.1 and will be removed in 0.3.0. Use :meth:`~langchain_openai.chat_models.base.ChatOpenAI.bind_tools` instead.
  | llm.bind_functions(functions=[function_def], function_call="route")


### Construct Graph

We're ready to start building the graph. Below, define the state and worker nodes using the function we just defined.

In [ ]:
import functools
import operator
from typing import Sequence, TypedDict

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langgraph.graph import END, StateGraph, START


# The agent state is the input to each node in the graph
class AgentState(TypedDict):
    # The annotation tells the graph that new messages will always
    # be added to the current states
    messages: Annotated[Sequence[BaseMessage], operator.add]
    # The 'next' field indicates where to route to next
    next: str

In [ ]:
@tool
def get_similar_requirements(requirement: str) -> dict[(str, float)]:
    """
    Given an input requirement as `requirement`, performs a similarity search over all
    available requirements.

    Returns the most similar artifacts matching `query` as tuples of the requirement
    itself and the similarity score.
    """
    artifacts = ["This is requirement " + str(index + 1) for index in range(10)]

    num_search_results = random.randint(1, len(artifacts))
    search_result_indexes = random.sample(range(len(artifacts)), num_search_results)

    search_results = [
        artifact for index, artifact in enumerate(artifacts)
        if index in search_result_indexes
    ]

    return [(artifact, random.uniform(0., 1.)) for artifact in search_results]

In [ ]:

similarity_agent = create_agent(
    llm,
    [get_similar_requirements],
    " similarity search over allavailable requirements.",
)
similarity_agent_node = functools.partial(agent_node, agent=similarity_agent, name="similarity_agent")

In [ ]:
@tool
def get_weather(location: str) -> dict:
    """Gives us the current weather information.

    Arguments:
        location -- the city the weather is requested for.

    Returns a dictionary with the following keys:
        temperature -- the temperature in degrees.
        unit -- `celsius` or `fahrenheit`.
        wind -- the wind strength; one of ['none', 'light', 'normal', 'strong']
        rain -- the chance of rain on a 0--10 scale; the number indicates the
            likelihood of rain, i.e., 0 = 0% chance of rain; 10 = 100% chance of rain.
    """
    return {
        "temperature": random.randint(7, 35),
        "unit": "celcius",
        "wind": random.choice(["none", "mild", "strong"]),
        "rain": random.randint(0, 100),
    }

In [ ]:
weather_agent = create_agent(
    llm,
    [get_weather],
    "You are responsible for find out weather condition",
)
weather_agent_node = functools.partial(agent_node, agent=weather_agent, name="weather_agent")

In [ ]:
workflow = StateGraph(AgentState)
workflow.add_node("weather_agent", weather_agent_node)
workflow.add_node("similarity_agent", similarity_agent_node)
workflow.add_node("supervisor", supervisor_chain)

In [ ]:
# Now connect all the edges in the graph.
for member in members:
    # We want our workers to ALWAYS "report back" to the supervisor when done
    workflow.add_edge(member, "supervisor")
# The supervisor populates the "next" field in the graph state
# which routes to a node or finishes
conditional_map = {k: k for k in members}
conditional_map["FINISH"] = END
workflow.add_conditional_edges("supervisor", lambda x: x["next"], conditional_map)
# Finally, add entrypoint
workflow.add_edge(START, "supervisor")

graph = workflow.compile()

### Invoke the team

With the graph created, we can now invoke it and see how it performs!

In [ ]:
for s in graph.stream(
    {
        "messages": [
            HumanMessage(content="find out weather information in Bengaluru")
        ]
    }
):
    if "__end__" not in s:
        print(s)
        print("----")

{'supervisor': {'next': 'weather_agent'}}
----
{'weather_agent': {'messages': [HumanMessage(content='The current weather in Bengaluru is as follows:\n\n- **Temperature:** 10°C\n- **Wind:** Mild\n- **Chance of Rain:** 39 (on a scale of 0-10, indicating a moderate likelihood of rain)', additional_kwargs={}, response_metadata={}, name='weather_agent')]}}
----
{'supervisor': {'next': 'similarity_agent'}}
----
{'similarity_agent': {'messages': [HumanMessage(content='The current weather in Bengaluru is:\n\n- **Temperature:** 10°C\n- **Wind:** Mild\n- **Chance of Rain:** 39 (on a scale of 0-10, indicating a moderate likelihood of rain)', additional_kwargs={}, response_metadata={}, name='similarity_agent')]}}
----
{'supervisor': {'next': 'FINISH'}}
----


In [ ]:
for s in graph.stream(
    {"messages": [HumanMessage(content="find out similar requirements in break systems.")]},
    {"recursion_limit": 100},
):
    if "__end__" not in s:
        print(s)
        print("----")

{'supervisor': {'next': 'similarity_agent'}}
----
{'similarity_agent': {'messages': [HumanMessage(content='Here are the similar requirements related to break systems along with their similarity scores:\n\n1. **Requirement:** This is requirement 2 - **Score:** 0.90\n2. **Requirement:** This is requirement 9 - **Score:** 0.83\n3. **Requirement:** This is requirement 3 - **Score:** 0.69\n4. **Requirement:** This is requirement 10 - **Score:** 0.70\n5. **Requirement:** This is requirement 6 - **Score:** 0.73\n6. **Requirement:** This is requirement 8 - **Score:** 0.24\n7. **Requirement:** This is requirement 7 - **Score:** 0.03\n\nThe higher the score, the more similar the requirement is to the break systems.', additional_kwargs={}, response_metadata={}, name='similarity_agent')]}}
----
{'supervisor': {'next': 'FINISH'}}
----


In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    # This requires some extra dependencies and is optional
    print(e)

Could not find a suitable TLS CA certificate bundle, invalid path: /home/jovyan/work/work/certificates.pem


# References

- [NVIDIA Generative AI Glossary](https://www.nvidia.com/en-us/glossary/generative-ai/)
- [Microsoft Azure Prompt Flow Overview](https://learn.microsoft.com/en-us/azure/machine-learning/prompt-flow/overview-what-is-prompt-flow?view=azureml-api-2)
- Alto, V. (2024). *Building LLM Powered Applications*. O'Reilly Media. [O'Reilly Link](https://learning.oreilly.com/library/view/building-llm-powered/9781835462317/Text/Chapter_04.xhtml#_idParaDest-56)
- Alammar, J., & Grootendorst, M. (2023). *Hands-On Large Language Models*. O'Reilly Media. [O'Reilly Link](https://learning.oreilly.com/library/view/hands-on-large-language/9781098150952/)
- Phoenix, J., & Taylor, M. (2023). *Prompt Engineering for Generative AI*. O'Reilly Media. [O'Reilly Link](https://learning.oreilly.com/library/view/prompt-engineering-for/9781098153427/)
- [LangChain Python Documentation](https://python.langchain.com/v0.2/docs/introduction/)
- [Prompting Guide](https://www.promptingguide.ai/techniques)
- Brown, T. B., Mann, B., Ryder, N., Subbiah, M., Kaplan, J., Dhariwal, P., Neelakantan, A., Shyam, P., Sastry, G., Askell, A., Agarwal, S., Herbert-Voss, A., Krueger, G., Henighan, T., Child, R., Ramesh, A., Ziegler, D. M., Wu, J., Winter, C., Hesse, C., Chen, M., Sigler, E., Litwin, M., Gray, S., Chess, B., Clark, J., Berner, C., McCandlish, S., Radford, A., Sutskever, I., & Amodei, D. (2020). *Language Models are Few-Shot Learners*. arXiv preprint arXiv:2005.14165. [arXiv Link](https://arxiv.org/abs/2005.14165)
- Wei, J., Wang, X., Schuurmans, D., Bosma, M., Ichter, B., Xia, F., Chi, E., Le, Q., & Zhou, D. (2023). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models*. arXiv preprint arXiv:2201.11903. [arXiv Link](https://arxiv.org/abs/2201.11903)
- [Master Prompting Concepts: Chain of Thought Prompting (promptengineering.org)](https://promptengineering.org)
- Gonen, H., et al. (2022). *Demystifying Prompts in Language Models via Perplexity Estimation*.
- [GPT-4 Open AI ChatGPT News (Global News)](https://globalnews.ca/news/9553331/gpt-4-open-ai-chatgpt/)
- [LangChain Multi-Agent Supervisor Tutorial](https://github.com/langchain-ai/langgraph/blob/main/docs/docs/tutorials/multi_agent/agent_supervisor.ipynb)
